In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_timestamp
from datetime import datetime

In [3]:
# Initialize Spark Session
from pyspark.sql import SparkSession

# Stop any existing SparkSession
if "spark" in locals():
    spark.stop()

# Initialize Spark Session
spark = SparkSession.builder.appName("SyntheticUserActivity").getOrCreate()

print("SparkSession created successfully!") # Add this line to see if it gets there

25/04/07 17:29:32 WARN Utils: Your hostname, Nikhils-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.0.90 instead (on interface en0)
25/04/07 17:29:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/07 17:29:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/04/07 17:29:32 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession created successfully!


In [5]:
# Synthetic data as a list of tuples
data = [
    ('user1', datetime(2025, 4, 7, 10, 0, 0), 'view', 'itemA', 'session1'),
    ('user1', datetime(2025, 4, 7, 10, 0, 15), 'view', 'itemB', 'session1'),
    ('user1', datetime(2025, 4, 7, 10, 0, 30), 'click', 'itemA', 'session1'),
    ('user1', datetime(2025, 4, 7, 10, 1, 0), 'purchase', 'itemC', 'session1'),
    ('user1', datetime(2025, 4, 7, 10, 5, 0), 'view', 'itemA', 'session2'),
    ('user1', datetime(2025, 4, 7, 10, 5, 30), 'like', 'itemD', 'session2'),
    ('user2', datetime(2025, 4, 7, 10, 10, 0), 'view', 'itemE', 'session3'),
    ('user2', datetime(2025, 4, 7, 10, 10, 45), 'click', 'itemE', 'session3'),
    ('user2', datetime(2025, 4, 7, 10, 11, 30), 'view', 'itemF', 'session3'),
    ('user2', datetime(2025, 4, 7, 10, 12, 0), 'purchase', 'itemG', 'session3'),
    ('user2', datetime(2025, 4, 7, 10, 12, 10), 'purchase', 'itemH', 'session3'),
    ('user3', datetime(2025, 4, 7, 10, 15, 0), 'view', 'itemA', 'session4'),
    ('user3', datetime(2025, 4, 7, 10, 15, 5), 'view', 'itemA', 'session4'),
    ('user3', datetime(2025, 4, 7, 10, 15, 10), 'view', 'itemB', 'session4'),
    ('user3', datetime(2025, 4, 7, 10, 20, 0), 'click', 'itemC', 'session5'),
    ('user3', datetime(2025, 4, 7, 10, 20, 30), 'like', 'itemD', 'session5'),
    ('user1', datetime(2025, 4, 7, 10, 25, 0), 'view', 'itemB', 'session1'),
    ('user2', datetime(2025, 4, 7, 10, 30, 0), 'view', 'itemE', 'session4'),
    ('user1', datetime(2025, 4, 7, 10, 35, 0), 'purchase', 'itemI', 'session6'),
    ('user1', datetime(2025, 4, 7, 10, 35, 10), 'purchase', 'itemJ', 'session6'),
    ('user1', datetime(2025, 4, 7, 10, 35, 30), 'view', 'itemK', 'session6'),
    ('user2', datetime(2025, 4, 7, 10, 40, 0), 'like', 'itemL', 'session5'),
    ('user2', datetime(2025, 4, 7, 10, 44, 0), 'purchase', 'itemM', 'session5'),
    ('user2', datetime(2025, 4, 7, 10, 44, 30), 'view', 'itemN', 'session5'),
    ('user2', datetime(2025, 4, 7, 10, 44, 50), 'view', 'itemO', 'session5'),
    ('user2', datetime(2025, 4, 7, 10, 45, 10), 'purchase', 'itemP', 'session5'),
    ('user3', datetime(2025, 4, 7, 10, 50, 0), 'view', 'itemQ', 'session6'),
    ('user3', datetime(2025, 4, 7, 10, 50, 5), 'click', 'itemQ', 'session6'),
    ('user3', datetime(2025, 4, 7, 10, 54, 30), 'purchase', 'itemR', 'session6'),
    ('user3', datetime(2025, 4, 7, 10, 54, 45), 'view', 'itemS', 'session6'),
    ('user3', datetime(2025, 4, 7, 10, 55, 0), 'purchase', 'itemT', 'session6')
]

# Define the schema
schema = ["user_id", "activity_timestamp", "activity_type", "item_id", "session_id"]

# Create the DataFrame
user_activity_df = spark.createDataFrame(data, schema=schema)

# Show the DataFrame
user_activity_df.show()
user_activity_df.printSchema()

# You can now use this DataFrame to apply your PySpark window functions.

# Don't forget to stop the SparkSession when you're done
# spark.stop()

+-------+-------------------+-------------+-------+----------+
|user_id| activity_timestamp|activity_type|item_id|session_id|
+-------+-------------------+-------------+-------+----------+
|  user1|2025-04-07 10:00:00|         view|  itemA|  session1|
|  user1|2025-04-07 10:00:15|         view|  itemB|  session1|
|  user1|2025-04-07 10:00:30|        click|  itemA|  session1|
|  user1|2025-04-07 10:01:00|     purchase|  itemC|  session1|
|  user1|2025-04-07 10:05:00|         view|  itemA|  session2|
|  user1|2025-04-07 10:05:30|         like|  itemD|  session2|
|  user2|2025-04-07 10:10:00|         view|  itemE|  session3|
|  user2|2025-04-07 10:10:45|        click|  itemE|  session3|
|  user2|2025-04-07 10:11:30|         view|  itemF|  session3|
|  user2|2025-04-07 10:12:00|     purchase|  itemG|  session3|
|  user2|2025-04-07 10:12:10|     purchase|  itemH|  session3|
|  user3|2025-04-07 10:15:00|         view|  itemA|  session4|
|  user3|2025-04-07 10:15:05|         view|  itemA|  se

In [84]:
from pyspark.sql.window import Window
from pyspark.sql.functions import first, col, hour, minute, second, sum

In [9]:
# # Imagine we want to understand user 
# behavior within their sessions. Specifically, for each user and each session, we want to find:

# # The time difference between each activity and the first activity 
# in that session. This will help us see how long a user spends between 
# different actions within a session.
# # The total number of 'view' events in that session up to each activity.
# This can show us how many items a user has viewed so far in their current session.

In [13]:
window_spec = Window.partitionBy("user_id","session_id").orderBy("activity_timestamp")

In [21]:
df = user_activity_df.withColumn("TimeDiff", first("activity_timestamp").over(window_spec))

In [43]:
df_time_duration = df.withColumn("Duration",col("activity_timestamp").cast("long")- col("TimeDiff").cast("long"))

In [45]:
df_time_duration.show()

+-------+-------------------+-------------+-------+----------+-------------------+--------+
|user_id| activity_timestamp|activity_type|item_id|session_id|           TimeDiff|Duration|
+-------+-------------------+-------------+-------+----------+-------------------+--------+
|  user1|2025-04-07 10:00:00|         view|  itemA|  session1|2025-04-07 10:00:00|       0|
|  user1|2025-04-07 10:00:15|         view|  itemB|  session1|2025-04-07 10:00:00|      15|
|  user1|2025-04-07 10:00:30|        click|  itemA|  session1|2025-04-07 10:00:00|      30|
|  user1|2025-04-07 10:01:00|     purchase|  itemC|  session1|2025-04-07 10:00:00|      60|
|  user1|2025-04-07 10:25:00|         view|  itemB|  session1|2025-04-07 10:00:00|    1500|
|  user1|2025-04-07 10:05:00|         view|  itemA|  session2|2025-04-07 10:05:00|       0|
|  user1|2025-04-07 10:05:30|         like|  itemD|  session2|2025-04-07 10:05:00|      30|
|  user1|2025-04-07 10:35:00|     purchase|  itemI|  session6|2025-04-07 10:35:0

In [62]:
df_cum_view = df_time_duration.withColumn("View", (col("activity_type")=="view").cast("Integer"))

In [64]:
df_cum_view.show()

+-------+-------------------+-------------+-------+----------+-------------------+--------+----+
|user_id| activity_timestamp|activity_type|item_id|session_id|           TimeDiff|Duration|View|
+-------+-------------------+-------------+-------+----------+-------------------+--------+----+
|  user1|2025-04-07 10:00:00|         view|  itemA|  session1|2025-04-07 10:00:00|       0|   1|
|  user1|2025-04-07 10:00:15|         view|  itemB|  session1|2025-04-07 10:00:00|      15|   1|
|  user1|2025-04-07 10:00:30|        click|  itemA|  session1|2025-04-07 10:00:00|      30|   0|
|  user1|2025-04-07 10:01:00|     purchase|  itemC|  session1|2025-04-07 10:00:00|      60|   0|
|  user1|2025-04-07 10:25:00|         view|  itemB|  session1|2025-04-07 10:00:00|    1500|   1|
|  user1|2025-04-07 10:05:00|         view|  itemA|  session2|2025-04-07 10:05:00|       0|   1|
|  user1|2025-04-07 10:05:30|         like|  itemD|  session2|2025-04-07 10:05:00|      30|   0|
|  user1|2025-04-07 10:35:00| 

In [76]:
df_cum_view.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- activity_timestamp: timestamp (nullable = true)
 |-- activity_type: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- TimeDiff: timestamp (nullable = true)
 |-- Duration: long (nullable = true)
 |-- View: integer (nullable = true)



In [78]:
window_spec2 = Window.partitionBy("user_id","session_id","item_id").orderBy("activity_timestamp")

In [86]:
df_cum_view_cnt = df_cum_view.withColumn("CntView", sum(col("View")).over(window_spec2))

In [88]:
df_cum_view_cnt.show()

+-------+-------------------+-------------+-------+----------+-------------------+--------+----+-------+
|user_id| activity_timestamp|activity_type|item_id|session_id|           TimeDiff|Duration|View|CntView|
+-------+-------------------+-------------+-------+----------+-------------------+--------+----+-------+
|  user1|2025-04-07 10:00:00|         view|  itemA|  session1|2025-04-07 10:00:00|       0|   1|      1|
|  user1|2025-04-07 10:00:30|        click|  itemA|  session1|2025-04-07 10:00:00|      30|   0|      1|
|  user1|2025-04-07 10:00:15|         view|  itemB|  session1|2025-04-07 10:00:00|      15|   1|      1|
|  user1|2025-04-07 10:25:00|         view|  itemB|  session1|2025-04-07 10:00:00|    1500|   1|      2|
|  user1|2025-04-07 10:01:00|     purchase|  itemC|  session1|2025-04-07 10:00:00|      60|   0|      0|
|  user1|2025-04-07 10:05:00|         view|  itemA|  session2|2025-04-07 10:05:00|       0|   1|      1|
|  user1|2025-04-07 10:05:30|         like|  itemD|  se